# Technique 2 — Being Specific

Leaving output open to interpretation gets you inconsistent results. Spell out what you want and Claude has a clear target. Two flavors:

- **Output-quality guidelines** — what the *result* must have: length, structure/format, required elements, tone. **Use these in almost every prompt.**
- **Process steps** — how Claude should *think*: brainstorm → pick → outline → refine. Reserve for complex/analytical tasks (troubleshooting, decision-making, multi-angle analysis).

For the meal planner, adding six **output-quality guidelines** is enough. The course reports the score jumping **3.92 → 7.86** — roughly doubling — from this one change.

> Incremental: we keep the clear-and-direct opening and add a `Guidelines:` block. Inputs stay a plain bullet list — **XML-tag structure is the next technique**, so we isolate it there. Same harness, same `dataset.json`.

## Load the harness + existing dataset

In [1]:
import sys
import os
from dotenv import find_dotenv

REPO_ROOT = os.path.dirname(find_dotenv())
SECTION = os.path.join(REPO_ROOT, "building-with-the-claude-api", "03-prompt-engineering")
if SECTION not in sys.path:
    sys.path.insert(0, SECTION)

from prompt_evaluator import PromptEvaluator, add_user_message, chat

DATASET = os.path.join(SECTION, "dataset.json")
evaluator = PromptEvaluator(max_concurrent_tasks=3)
print("Harness loaded. Reusing dataset ->", DATASET)

Harness loaded. Reusing dataset -> c:\Development\anthropic-cert\building-with-the-claude-api\03-prompt-engineering\dataset.json


## Improved prompt — add output-quality guidelines

The six guidelines directly map onto the mandatory `extra_criteria` the grader enforces (caloric total, macros, exact foods/portions/timing) — which is exactly why the score leaps.

In [2]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## Re-evaluate

Same dataset, same criteria. The course saw ~3.9 → ~7.9 here — but on **Haiku 4.5** the earlier steps already score 8+, so locally expect the score to be high already and this change to hold it near the top rather than double it. The technique's value is real; the model just leaves little headroom to show it. (See the README's "capability drift" note.)

In [3]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file=DATASET,
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file=os.path.join(SECTION, "output-03-being-specific.json"),
    html_output_file=os.path.join(SECTION, "output-03-being-specific.html"),
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 8.333333333333334


## Takeaway

Telling Claude exactly which elements to include is the highest-leverage change so far — it aligns the output directly with the grading criteria. **Output-quality guidelines belong in almost every prompt.** For harder, analytical tasks you'd *also* add **process steps** (brainstorm → choose → outline → refine) to make Claude reason through multiple angles before answering — not needed for this straightforward generation task.

Next: **structure with XML tags** — organizing the prompt so Claude cleanly separates the athlete's data from the instructions.